<a href="https://colab.research.google.com/github/giumont/overlap_resolver/blob/main/notebooks/overlap_pairs_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pair-level preprocessing for HH->bbtautau: (jet, tau) overlap dataset from .root files

This notebook builds the labelled, preprocessed **train / validation / test** datasets for the objective 3.1 pair-classification task (FF / FT / TF / TT), starting directly from the `.root` ntuples in `overlap_resolver/root_datasets/HH_bbtt`.

It reuses:
- `flavour_tag_ml` (repo `flavour_tagging`): generic checkpoint I/O and standardization utilities (`fit_transform_standardize`, `transform_standardize`, `compute_feature_stats`, `JetDataset`, ...).
- `overlap_resolver/src`: `obj_3_1.py` (file loading + analysis-level selection), `overlap_kinematics.py` (pair kinematics), `truth_vs_reco_params.py` (truth labelling), and `overlap_pairs_dataset_builder.py` (this notebook's main orchestrator, `build_pair_dataset_from_root` + `split_pairs_by_event`).

Pipeline:
1. **STEP 1** — build the (jet, tau) pair dataset from all `.root` files, with geometric overlap slicing and truth labelling, checkpointed chunk by chunk.
2. **STEP 2** — merge the checkpoint chunks into a single (X, y, event_id) array.
3. **STEP 3** — split into train / val / test **by event** (no data leakage).
4. **STEP 4** — materialize each split to disk.
5. **STEP 5** — standardize features (fit on train, transform val/test).
6. **STEP 6** — final sanity checks (shapes, 4-class balance, NaN/Inf, mean/std).
7. *(Optional)* — wrap the splits in `JetDataset` / `DataLoader`.


## 0. Setup

In [1]:
# MOUNT GOOGLE DRIVE
#
# Used ONLY to persist the produced datasets (X/y/event_id splits,
# normalization params, ...) across Colab sessions. The .root INPUT files
# are not read from Drive: they come from the cloned "overlap_resolver" repo.

from google.colab import drive
drive.mount("/content/drive/")


Mounted at /content/drive/


In [2]:
# IMPORT GITHUB REPO: flavour_tag_ml (repo "flavour_tagging")
#
# Contains generic, reusable pieces of the pipeline: checkpoint_io.py,
# merge_datasets.py, data.py (JetDataset, standardization helpers), utils.py.

from google.colab import userdata
import os
import sys

token = userdata.get("GITHUB_TOKEN")

REPO_PATH_FLAVOUR_TAG_ML = "/content/flavour_tagging"

if not os.path.exists(REPO_PATH_FLAVOUR_TAG_ML):
    !git clone https://{token}@github.com/giumont/flavour_tagging.git {REPO_PATH_FLAVOUR_TAG_ML}
else:
    !git -C {REPO_PATH_FLAVOUR_TAG_ML} pull --rebase

# The importable package lives under <repo>/src/flavour_tag_ml, so we add
# <repo>/src (not the repo root) to sys.path and import it as a top-level
# package: "import flavour_tag_ml".
SRC_PATH_FLAVOUR_TAG_ML = os.path.join(REPO_PATH_FLAVOUR_TAG_ML, "src")
if SRC_PATH_FLAVOUR_TAG_ML not in sys.path:
    sys.path.append(SRC_PATH_FLAVOUR_TAG_ML)

import flavour_tag_ml


Cloning into '/content/flavour_tagging'...
remote: Enumerating objects: 938, done.
remote: Counting objects: 100% (123/123), done.
remote: Compressing objects: 100% (80/80), done.
remote: Total 938 (delta 59), reused 76 (delta 32), pack-reused 815 (from 2)
Receiving objects: 100% (938/938), 156.49 MiB | 17.05 MiB/s, done.
Resolving deltas: 100% (494/494), done.


In [3]:
# IMPORT GITHUB REPO: overlap_resolver
#
# Contains, under "src/": obj_3_1.py, overlap_kinematics.py,
# truth_vs_reco_params.py and overlap_pairs_dataset_builder.py (this
# notebook's main orchestrator). The input .root files live under
# "root_datasets/HH_bbtt/" in this same repo. This is also, under
# "notebooks/", where this notebook itself is meant to live.
#
# NOTE: the original snippet reassigned "repo_path" to the SAME path used
# for flavour_tagging ("/content/flavour_tagging") -- fixed here: this repo
# gets its own clone path, REPO_PATH_OVERLAP_RESOLVER.

token_overlap = userdata.get("GITHUB_TOKEN_OVERLAP_RESOLVER")

REPO_PATH_OVERLAP_RESOLVER = "/content/overlap_resolver"

if not os.path.exists(REPO_PATH_OVERLAP_RESOLVER):
    !git clone https://{token_overlap}@github.com/giumont/overlap_resolver.git {REPO_PATH_OVERLAP_RESOLVER}
else:
    !git -C {REPO_PATH_OVERLAP_RESOLVER} pull --rebase

# Unlike flavour_tag_ml, the modules under overlap_resolver/src are FLAT:
# they import each other directly (e.g. overlap_kinematics.py does
# "from obj_3_1 import ..." and "from truth_vs_reco_params import ...",
# NOT "from src.obj_3_1 import ..."). To keep them working unmodified, we
# append the "src" folder ITSELF to sys.path (not the repo root), and
# import them below as flat top-level modules.
SRC_PATH_OVERLAP_RESOLVER = os.path.join(REPO_PATH_OVERLAP_RESOLVER, "src")
if SRC_PATH_OVERLAP_RESOLVER not in sys.path:
    sys.path.append(SRC_PATH_OVERLAP_RESOLVER)


Cloning into '/content/overlap_resolver'...
remote: Enumerating objects: 884, done.
remote: Counting objects: 100% (80/80), done.
remote: Compressing objects: 100% (52/52), done.
remote: Total 884 (delta 31), reused 68 (delta 20), pack-reused 804 (from 1)
Receiving objects: 100% (884/884), 708.12 MiB | 19.82 MiB/s, done.
Resolving deltas: 100% (321/321), done.
Updating files: 100% (549/549), done.


In [4]:
# INSTALL EXTRA DEPENDENCIES (not preinstalled on Colab)
#
# awkward / uproot: needed by obj_3_1.py, overlap_kinematics.py and
# truth_vs_reco_params.py to read the .root ntuples and handle jagged
# (variable-length per-event) arrays.
!pip install -q awkward uproot

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 57.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 698.7/698.7 kB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 405.8/405.8 kB 28.7 MB/s eta 0:00:00


In [5]:
# IMPORT PROJECT FUNCTIONS

# --- flavour_tag_ml: proper package, "from flavour_tag_ml.<module> import ..." ---
from flavour_tag_ml.data import (
    JetDataset,
    fit_transform_standardize,
    transform_standardize,
    compute_feature_stats,
)
import flavour_tag_ml.checkpoint_io as ft_checkpoint_io  # kept for reference/debug only
from flavour_tag_ml.merge_datasets import merge_checkpoint_chunks, check_checkpoint_progress
from flavour_tag_ml.utils import set_seed, load_mmap

# --- overlap_resolver/src: FLAT modules (see note in the cell above) ---
import obj_3_1
import overlap_kinematics
import truth_vs_reco_params
import overlap_pairs_dataset_builder as pair_builder

In [ ]:
# RELOAD MODULES
#
# Convenience for development only (picks up edits made to the .py files
# during this Colab session); harmless, and a no-op in effect, otherwise.
# Reload order matters: truth_vs_reco_params is reloaded BEFORE
# overlap_kinematics, since the latter imports names FROM the former at
# import time (see overlap_kinematics.py top-level imports).

import importlib

importlib.reload(flavour_tag_ml.data)
importlib.reload(flavour_tag_ml.checkpoint_io)
importlib.reload(flavour_tag_ml.merge_datasets)
importlib.reload(flavour_tag_ml.utils)

importlib.reload(obj_3_1)
importlib.reload(truth_vs_reco_params)
importlib.reload(overlap_kinematics)
importlib.reload(pair_builder)


<module 'overlap_pairs_dataset_builder' from '/content/overlap_resolver/src/overlap_pairs_dataset_builder.py'>

In [6]:
# STANDARD PYTHON LIBRARIES

import gc
import shutil
from pathlib import Path

import numpy as np
import awkward as ak
import torch
from torch.utils.data import DataLoader


In [7]:
# (OPTIONAL) Manual inspection of one .root file
#
# Lists the branches available in the TTree, purely as a sanity check
# before filling in the branch names in GLOBAL PARAMS below.
# Set CHECK_FILE = None to skip this cell entirely.

import uproot

CHECK_FILE = os.path.join(REPO_PATH_OVERLAP_RESOLVER, "root_datasets", "bkg_tt",
                 "output_TTBAR_mc23a_bypass_noOR_000001.root")

CHECK_TREE_NAME = "AnalysisMiniTree"

if CHECK_FILE is not None:
    with uproot.open(CHECK_FILE) as root_file:
        tree = root_file[CHECK_TREE_NAME]
        branch_names = sorted(tree.keys())
        print(f"Tree: {CHECK_TREE_NAME}  |  entries: {tree.num_entries}  |  branches: {len(branch_names)}")
        for name in branch_names:
            print(" ", name)


Tree: AnalysisMiniTree  |  entries: 9679  |  branches: 369
  GRL2022
  GRL2022_ignore_TRIGLAR
  PileupWeight___NOSYS
  RandomLumiBlockNumber
  RandomRunNumber
  actualInteractionsPerCrossing
  averageInteractionsPerCrossing
  bbtt_HH_delta_phi___NOSYS
  bbtt_HH_eta___NOSYS
  bbtt_HH_m___NOSYS
  bbtt_HH_phi___NOSYS
  bbtt_HH_pt___NOSYS
  bbtt_HH_vis_delta_phi___NOSYS
  bbtt_HH_vis_eta___NOSYS
  bbtt_HH_vis_m___NOSYS
  bbtt_HH_vis_phi___NOSYS
  bbtt_HH_vis_pt___NOSYS
  bbtt_H_bb_eta___NOSYS
  bbtt_H_bb_m___NOSYS
  bbtt_H_bb_phi___NOSYS
  bbtt_H_bb_pt___NOSYS
  bbtt_H_vis_tautau_eta___NOSYS
  bbtt_H_vis_tautau_m___NOSYS
  bbtt_H_vis_tautau_phi___NOSYS
  bbtt_H_vis_tautau_pt___NOSYS
  bbtt_Jet_b1_E___NOSYS
  bbtt_Jet_b1_eta___NOSYS
  bbtt_Jet_b1_muonCorrPt___NOSYS
  bbtt_Jet_b1_nmuons___NOSYS
  bbtt_Jet_b1_pcbt_GN2v01___NOSYS
  bbtt_Jet_b1_phi___NOSYS
  bbtt_Jet_b1_pt___NOSYS
  bbtt_Jet_b1_truthLabel___NOSYS
  bbtt_Jet_b1_uncorrPt___NOSYS
  bbtt_Jet_b2_E___NOSYS
  bbtt_Jet_b2_eta___NOSYS
 

## 1. Global parameters

All configurable constants in one block, ALL_CAPS, English comments.

**Branches flagged `# TODO CONFIRM`**: I could not verify these exact branch names against the actual `.root` files (no ROOT file / uproot listing was available to me), so they're best-effort guesses based on naming conventions in the other scripts. Please run the optional "manual inspection" cell above (set `CHECK_FILE` to a real path) and confirm/correct them before running STEP 1 for real — a wrong branch name will simply raise a `KeyError` in `tree.arrays(...)`, so it's safe but wasteful to run with an unconfirmed name.

In [12]:
# =============================================================================
# GLOBAL PARAMETERS
# =============================================================================

# --- Paths ----------------------------------------------------------------
ROOT_INPUT_DIR = os.path.join(REPO_PATH_OVERLAP_RESOLVER, "root_datasets", "bkg_tt")

DATASETS_PATH = "/content/drive/MyDrive/overlap_resolver/datas/np_arrays/bkg_tt"
os.makedirs(DATASETS_PATH, exist_ok=True)

# --- obj_3_1.py module-level attributes ---
obj_3_1.ROOT_DIR = Path(ROOT_INPUT_DIR)
obj_3_1.FILE_PREFIX = "output_TTBAR_mc23a_bypass_noOR_0000"
obj_3_1.FILE_SUFFIX = ".root"
obj_3_1.TREE_NAME = "AnalysisMiniTree"
obj_3_1.N_ENTRIES_CAP = None

# --- Analysis-level selection branches ---
JET_ANALYSIS_BRANCH = "recojet_antikt4PFlow_isAnalysisJet___NOSYS"
TAU_ANALYSIS_BRANCH = "tau_isAnalysisTau___NOSYS"

# --- Core kinematic branches ---
JET_ETA_BRANCH = "recojet_antikt4PFlow_eta"
JET_PHI_BRANCH = "recojet_antikt4PFlow_phi"
JET_PT_BRANCH = "recojet_antikt4PFlow_pt___NOSYS"

TAU_ETA_BRANCH = "tau_eta"
TAU_PHI_BRANCH = "tau_phi"
TAU_PT_BRANCH = "tau_pt___NOSYS"

# --- MET branches ---
MET_BRANCH = "met_met___NOSYS"
MET_PHI_BRANCH = "met_phi___NOSYS"

# --- Feature Flags ---
COMPUTE_PT_RATIO = True
COMPUTE_MET_PROJ = True
COMPUTE_MT = True

# --- Extra branches aggregate (slicing & preprocessing passivo) ---
EXTRA_JET_BRANCHES = {
    "jet_mass": "recojet_antikt4PFlow_m___NOSYS",
    "jet_n_muons": "recojet_antikt4PFlow_n_muons___NOSYS",
    "jet_gn2_score": "recojet_antikt4PFlow_ftag_quantile_GN2v01_Continuous",
}

EXTRA_TAU_BRANCHES = {
    "tau_nProng": "tau_nProng",
    "tau_decayMode": "tau_decayMode",
    "tau_charge": "tau_charge",
    "tau_gn_score": "tau_GNTauScoreSigTrans_v0prune",
    "tau_rnn_jet_score": "tau_RNNJetScoreSigTrans",
    "tau_rnn_ele_score": "tau_RNNEleScoreSigTrans_v1",
}

# --- Truth-label branches ---
JET_TRUTH_LABEL_BRANCH = "recojet_antikt4PFlow_HadronConeExclTruthLabelID"
JET_TRUTH_LABEL_B_VALUE = 5
TAU_TRUTH_MATCH_BRANCH = "tau_truth_IsHadronicTau"

# --- Geometric overlap threshold ---
DR_THR = 0.4

# --- Feature columns of X ---
FEATURE_KEYS = [
    "pair_dr", "pair_deta", "pair_dphi", "pair_pt_ratio",
    "jet_pt", "jet_eta", "jet_phi", "jet_mass", "jet_n_muons", "jet_gn2_score",
    "tau_pt", "tau_eta", "tau_phi", "tau_nProng", "tau_decayMode", "tau_charge",
    "tau_gn_score", "tau_rnn_jet_score", "tau_rnn_ele_score",
    "tau_met_proj", "tau_mt",
]

# --- Pair truth-label encoding ---
LABEL_INDEX_MAP = {"FF": 0, "FT": 1, "TF": 2, "TT": 3}

# --- Train / val / test split fractions ---
TRAIN_FRAC = 0.70
VAL_FRAC = 0.15
TEST_FRAC = 0.15
assert abs(TRAIN_FRAC + VAL_FRAC + TEST_FRAC - 1.0) < 1e-8, "fractions must sum to 1"

RANDOM_SEED = 42

# --- Checkpoint base name ---
PAIR_SAVE_NAME = f"bkg_tt_pairs_DR={DR_THR}"

## 2. Set global seed

In [9]:
# SET GLOBAL SEED (reproducibility across numpy / torch / python `random`)
set_seed(RANDOM_SEED)


## 3. STEP 1 — Build the pair dataset from `.root` files (checkpointed)

**Caveat on "resuming"** (flagging this explicitly, since it differs from the H5 pipeline's checkpointing): `build_pair_dataset_from_root` always loops over **all** files returned by `load_files()` and writes chunks starting again from `chunk_idx=0` / `event_offset=0` — it does **not** detect an existing manifest and skip already-processed files. So re-running the cell below after an interruption will **overwrite** the previous checkpoint from scratch (each file is still fast to re-read, so for 14 files this is a correctness-over-efficiency tradeoff, not a bug) rather than resume mid-way. If you want true cross-session resumption I can add that, but it's not implemented in `overlap_pairs_dataset_builder.py` as given — let me know.

In [10]:
# =============================================================================
# STEP 1a — truth-label closures for jet / tau
#
# Parametric equivalent of label_jets_and_taus (truth_vs_reco_params.py),
# restricted to TRUTH_MODE_TAU="label" (the geometric-matching branch of the
# original function is not exposed here, since build_pair_dataset_from_root
# expects a single bool array per closure, not the (label, dr_truth) pair
# that the geometric mode returns).
#
# Required signature: fn(events_full, selection_mask) -> awkward.Array (bool)
# already indexed by the analysis-level selection mask, i.e. same jagged
# structure as jet_sel / tau_sel themselves.
# =============================================================================

def jet_truth_label_fn(a, jet_sel):
    """True jet = HadronConeExclTruthLabelID == JET_TRUTH_LABEL_B_VALUE (b-jet)."""
    flavour = a[JET_TRUTH_LABEL_BRANCH][jet_sel]
    return flavour == JET_TRUTH_LABEL_B_VALUE


def tau_truth_label_fn(a, tau_sel):
    """True tau = tau_truth_IsHadronicTau != 0."""
    return a[TAU_TRUTH_MATCH_BRANCH][tau_sel] != 0


In [13]:
# =============================================================================
# STEP 1b — build the (jet, tau) pair dataset from all .root files
# =============================================================================

build_result = pair_builder.build_pair_dataset_from_root(
    root_dir=ROOT_INPUT_DIR,
    save_path=DATASETS_PATH,
    save_name=PAIR_SAVE_NAME,
    jet_analysis_branch=JET_ANALYSIS_BRANCH,
    tau_analysis_branch=TAU_ANALYSIS_BRANCH,
    jet_eta_branch=JET_ETA_BRANCH,
    jet_phi_branch=JET_PHI_BRANCH,
    jet_pt_branch=JET_PT_BRANCH,
    tau_eta_branch=TAU_ETA_BRANCH,
    tau_phi_branch=TAU_PHI_BRANCH,
    tau_pt_branch=TAU_PT_BRANCH,
    jet_truth_label_fn=jet_truth_label_fn,
    tau_truth_label_fn=tau_truth_label_fn,
    jet_truth_label_branch=JET_TRUTH_LABEL_BRANCH,
    tau_truth_label_branch=TAU_TRUTH_MATCH_BRANCH,
    extra_jet_branches=EXTRA_JET_BRANCHES,
    extra_tau_branches=EXTRA_TAU_BRANCHES,
    met_branch=MET_BRANCH,
    met_phi_branch=MET_PHI_BRANCH,
    compute_pt_ratio=COMPUTE_PT_RATIO,
    compute_met_proj=COMPUTE_MET_PROJ,
    compute_mt=COMPUTE_MT,
    feature_keys=FEATURE_KEYS,
    dr_thr=DR_THR,
    label_index_map=LABEL_INDEX_MAP,
    verbose=True,
)

build_result


FILE INPUT
[OK] /content/overlap_resolver/root_datasets/bkg_tt/output_TTBAR_mc23a_bypass_noOR_000001.root   entries=9679   usati=9679
[OK] /content/overlap_resolver/root_datasets/bkg_tt/output_TTBAR_mc23a_bypass_noOR_000002.root   entries=9591   usati=9591
[OK] /content/overlap_resolver/root_datasets/bkg_tt/output_TTBAR_mc23a_bypass_noOR_000003.root   entries=9764   usati=9764
[OK] /content/overlap_resolver/root_datasets/bkg_tt/output_TTBAR_mc23a_bypass_noOR_000004.root   entries=9817   usati=9817
[OK] /content/overlap_resolver/root_datasets/bkg_tt/output_TTBAR_mc23a_bypass_noOR_000005.root   entries=9753   usati=9753
[OK] /content/overlap_resolver/root_datasets/bkg_tt/output_TTBAR_mc23a_bypass_noOR_000006.root   entries=9751   usati=9751
[OK] /content/overlap_resolver/root_datasets/bkg_tt/output_TTBAR_mc23a_bypass_noOR_000007.root   entries=9698   usati=9698
[OK] /content/overlap_resolver/root_datasets/bkg_tt/output_TTBAR_mc23a_bypass_noOR_000008.root   entries=9833   usati=9833
[OK]

{'n_pairs_saved': 102468,
 'n_files_processed': 9,
 'manifest_file': '/content/drive/MyDrive/overlap_resolver/datas/np_arrays/bkg_tt/bkg_tt_pairs_DR=0.4_manifest.npz'}

In [14]:
# STEP 1c — checkpoint progress check (pair-specific: no a-priori n_total,
# so this only reports how many pairs/chunks have been saved so far)
progress = pair_builder.check_pair_checkpoint_progress(DATASETS_PATH, PAIR_SAVE_NAME)
progress


PAIR CHECKPOINT (COMPLETE): /content/drive/MyDrive/overlap_resolver/datas/np_arrays/bkg_tt/bkg_tt_pairs_DR=0.4_manifest.npz
CHECK - Coppie salvate: 102468 in 9 chunk(s)


{'n_done': 102468, 'n_chunks': 9, 'complete': True}

## 4. STEP 2 — Merge the checkpoint chunks

`merge_pair_checkpoint_chunks` (pair-specific equivalent of `merge_checkpoint_chunks` used in the H5 pipeline) reads back every per-file chunk written in STEP 1 plus the manifest, and returns the full, already-concatenated `X`, `y`, `event_id`, `feature_names` in memory.

**Scale assumption (point 6 from the proposal):** the (jet, tau) pair dataset is assumed small enough to fit in RAM after the merge (unlike the H5 jet pipeline, which needed memmap). If this turns out to be false for the full 14-file HH_bbtt sample, this step (and STEP 4 below) should be rewritten with `np.lib.format.open_memmap`, exactly as in the H5 notebook. `output_file=""` is passed to skip writing the (redundant) merged `.npz` to disk, since STEP 4 immediately re-saves the data split by train/val/test anyway.

In [15]:
# =============================================================================
# STEP 2 — Merge all per-file checkpoint chunks into a single in-memory
# (X, y, event_id, feature_names) tuple.
#
# output_file="" : skip writing a merged .npz to disk (STEP 4 below saves
#                   the train/val/test splits separately anyway, so an
#                   intermediate merged copy would just be redundant I/O).
# delete_chunks_after=False : keep the per-file chunks on disk, so STEP 1
#                   remains re-runnable/resumable without re-reading the
#                   .root files, in case the split/save logic below needs
#                   to be re-run from scratch.
# =============================================================================

X_all, y_all, event_id_all, feature_names = pair_builder.merge_pair_checkpoint_chunks(
    DATASETS_PATH,
    PAIR_SAVE_NAME,
    output_file="",
    delete_chunks_after=False,
    verbose=True,
)

print("\nfeature_names:", feature_names)


MERGING PAIR CHECKPOINT: /content/drive/MyDrive/overlap_resolver/datas/np_arrays/bkg_tt/bkg_tt_pairs_DR=0.4_manifest.npz
CHECK - X shape: (102468, 21)
CHECK - y shape: (102468,)
CHECK - event_id shape: (102468,)

feature_names: ['pair_dr', 'pair_deta', 'pair_dphi', 'pair_pt_ratio', 'jet_pt', 'jet_eta', 'jet_phi', 'jet_mass', 'jet_n_muons', 'jet_gn2_score', 'tau_pt', 'tau_eta', 'tau_phi', 'tau_nProng', 'tau_decayMode', 'tau_charge', 'tau_gn_score', 'tau_rnn_jet_score', 'tau_rnn_ele_score', 'tau_met_proj', 'tau_mt']


In [16]:
# STEP 2b — debug: overall shapes, per-class counts (FF/FT/TF/TT), unique events

print("X_all shape:", X_all.shape)
print("y_all shape:", y_all.shape)
print("event_id_all shape:", event_id_all.shape)

n_unique_events = len(np.unique(event_id_all))
print(f"\nUnique events represented in the pair dataset: {n_unique_events}")

INDEX_TO_LABEL = {v: k for k, v in LABEL_INDEX_MAP.items()}
print("\nPer-class pair counts:")
for idx in sorted(INDEX_TO_LABEL):
    count = int(np.sum(y_all == idx))
    frac = count / y_all.shape[0]
    print(f"  {INDEX_TO_LABEL[idx]:>2} (y={idx}): {count:>10}  ({100.0 * frac:.3f}%)")

assert set(np.unique(y_all)) <= set(LABEL_INDEX_MAP.values()), \
    "y_all contains label indices outside LABEL_INDEX_MAP!"


X_all shape: (102468, 21)
y_all shape: (102468,)
event_id_all shape: (102468,)

Unique events represented in the pair dataset: 62625

Per-class pair counts:
  FF (y=0):      76868  (75.017%)
  FT (y=1):         72  (0.070%)
  TF (y=2):      24896  (24.296%)
  TT (y=3):        632  (0.617%)


## 5. STEP 3 — Train / validation / test split without event-level data leakage

`split_pairs_by_event` splits on **unique `event_id` values**, not on individual pairs: every (jet, tau) pair belonging to the same event ends up in the same split. This differs from the H5 pipeline (which assigned whole *files* to a split): here, `build_pair_dataset_from_root` already merges all `.root` files into a single checkpoint, so the split happens directly on the merged pair dataset — no separate "file assignment" step is needed (point 5 from the proposal).

In [17]:
# =============================================================================
# STEP 3 — Split into train / val / test by EVENT (no pair from the same
# event can leak across splits).
# =============================================================================

train_idx, val_idx, test_idx = pair_builder.split_pairs_by_event(
    event_id_all,
    train_frac=TRAIN_FRAC,
    val_frac=VAL_FRAC,
    test_frac=TEST_FRAC,
    seed=RANDOM_SEED,
)

print(f"Pairs -> train: {train_idx.size} | val: {val_idx.size} | test: {test_idx.size}")
print(f"Total pairs accounted for: {train_idx.size + val_idx.size + test_idx.size} / {y_all.shape[0]}")


Pairs -> train: 71696 | val: 15334 | test: 15438
Total pairs accounted for: 102468 / 102468


In [18]:
# STEP 3b — sanity checks: no event_id leakage across splits + per-split
# class distribution.

train_events = set(np.unique(event_id_all[train_idx]).tolist())
val_events = set(np.unique(event_id_all[val_idx]).tolist())
test_events = set(np.unique(event_id_all[test_idx]).tolist())

assert train_events.isdisjoint(val_events), "LEAKAGE: events shared between train and val!"
assert train_events.isdisjoint(test_events), "LEAKAGE: events shared between train and test!"
assert val_events.isdisjoint(test_events), "LEAKAGE: events shared between val and test!"
print("[OK] no event_id overlap between train / val / test")

print(f"\nUnique events -> train: {len(train_events)} | val: {len(val_events)} | test: {len(test_events)} "
      f"| total: {len(train_events) + len(val_events) + len(test_events)} / {n_unique_events}")

for split_name, idx in (("TRAIN", train_idx), ("VAL", val_idx), ("TEST", test_idx)):
    print(f"\n--- {split_name} class distribution ---")
    y_split = y_all[idx]
    for label_idx in sorted(INDEX_TO_LABEL):
        count = int(np.sum(y_split == label_idx))
        frac = count / y_split.shape[0] if y_split.shape[0] else 0.0
        print(f"  {INDEX_TO_LABEL[label_idx]:>2}: {count:>10}  ({100.0 * frac:.3f}%)")


[OK] no event_id overlap between train / val / test

Unique events -> train: 43838 | val: 9394 | test: 9393 | total: 62625 / 62625

--- TRAIN class distribution ---
  FF:      53760  (74.983%)
  FT:         57  (0.080%)
  TF:      17444  (24.331%)
  TT:        435  (0.607%)

--- VAL class distribution ---
  FF:      11536  (75.232%)
  FT:          9  (0.059%)
  TF:       3698  (24.116%)
  TT:         91  (0.593%)

--- TEST class distribution ---
  FF:      11572  (74.958%)
  FT:          6  (0.039%)
  TF:       3754  (24.317%)
  TT:        106  (0.687%)


## 6. STEP 4 — Materialize the splits to disk

Each split is written as plain (uncompressed) `.npy` files for `X`/`y`/`event_id`, consistent with the H5 pipeline's convention (`np.save`, not `np.savez_compressed`, for the large arrays — cheap to load back with `load_mmap`/`np.load(..., mmap_mode="r")` later on). `feature_names` is saved once, since it's identical across splits.

In [19]:
# =============================================================================
# STEP 4 — Save X / y / event_id for each split as plain .npy files, plus
# feature_names once (identical across splits).
# =============================================================================

SPLITS = {"train": train_idx, "val": val_idx, "test": test_idx}

for split_name, idx in SPLITS.items():
    x_path = os.path.join(DATASETS_PATH, f"X_{split_name}.npy")
    y_path = os.path.join(DATASETS_PATH, f"y_{split_name}.npy")
    event_id_path = os.path.join(DATASETS_PATH, f"event_id_{split_name}.npy")

    np.save(x_path, X_all[idx])
    np.save(y_path, y_all[idx])
    np.save(event_id_path, event_id_all[idx])

    print(f"[OK] {split_name}: saved X{X_all[idx].shape}, y{y_all[idx].shape}, "
          f"event_id{event_id_all[idx].shape}")

np.savez_compressed(
    os.path.join(DATASETS_PATH, "pair_feature_names.npz"),
    feature_names=np.array(feature_names, dtype=object),
    label_index_map=np.array(list(LABEL_INDEX_MAP.items()), dtype=object),
    dr_thr=DR_THR,
)
print("\n[OK] feature_names / label_index_map / dr_thr saved to pair_feature_names.npz")


[OK] train: saved X(71696, 21), y(71696,), event_id(71696,)
[OK] val: saved X(15334, 21), y(15334,), event_id(15334,)
[OK] test: saved X(15438, 21), y(15438,), event_id(15438,)

[OK] feature_names / label_index_map / dr_thr saved to pair_feature_names.npz


In [20]:
# Free the large in-memory arrays before standardization (STEP 5): from
# here on, each split is reloaded from disk as needed, exactly as in the
# H5 pipeline's memory-management convention.

del X_all, y_all, event_id_all
gc.collect()


185985

## 7. STEP 5 — Standardize the features (fit on train, transform on val/test)

`fit_transform_standardize` / `transform_standardize` (`flavour_tag_ml.data`) are reused unmodified from the H5 pipeline: mean/std are computed **only on train**, in blocks, then applied (also blockwise) to val/test — val/test are never refit, to avoid leaking their distribution into the standardization parameters.

As already noted in STEP 2, the pair dataset is assumed small enough to be handled as plain in-RAM `ndarray`s (no `np.lib.format.open_memmap` staging needed, unlike the H5 pipeline): each split is simply reloaded from the `X_{split}.npy` / `y_{split}.npy` files written to disk in STEP 4 — since STEP 4 ended by freeing `X_all`/`y_all`/`event_id_all` from RAM — standardized, and saved back under a `_normalized` suffix.

In [21]:
# =============================================================================
# STEP 5a — Standardize TRAIN: fit mean/std on train and transform it.
#
# X_train is reloaded from disk (STEP 4 freed X_all/y_all/event_id_all from
# RAM) as a plain ndarray: fit_transform_standardize works on ndarray as well
# as memmap, so no memmap staging is required here.
# =============================================================================

X_train = np.load(os.path.join(DATASETS_PATH, "X_train.npy"))
y_train = np.load(os.path.join(DATASETS_PATH, "y_train.npy"))

X_train, mean, std = fit_transform_standardize(X_train, chunk_size=100_000, std_eps=1e-8, verbose=True)

np.save(os.path.join(DATASETS_PATH, "X_train_normalized.npy"), X_train)
np.save(os.path.join(DATASETS_PATH, "y_train_normalized.npy"), y_train)

np.savez_compressed(
    os.path.join(DATASETS_PATH, "pair_normalization_params.npz"),
    mean=mean,
    std=std,
    feature_names=np.array(feature_names, dtype=object),
)
print("[OK] train (normalized): saved X/y as .npy + mean/std/feature_names in pair_normalization_params.npz")

del X_train, y_train
gc.collect()


71,696/71,696
[OK] train (normalized): saved X/y as .npy + mean/std/feature_names in pair_normalization_params.npz


22

In [22]:
# =============================================================================
# STEP 5b — Standardize VAL with the TRAIN mean/std.
#
# mean/std are reloaded from the .npz saved in STEP 5a (robust to a runtime
# restart) rather than reusing the in-memory variables -- same
# robust-to-restart principle applied throughout the H5 pipeline.
# =============================================================================

norm_params = np.load(os.path.join(DATASETS_PATH, "pair_normalization_params.npz"), allow_pickle=True)
mean, std = norm_params["mean"], norm_params["std"]

X_val = np.load(os.path.join(DATASETS_PATH, "X_val.npy"))
y_val = np.load(os.path.join(DATASETS_PATH, "y_val.npy"))

X_val = transform_standardize(X_val, mean, std, chunk_size=100_000, std_eps=1e-8, verbose=True)

np.save(os.path.join(DATASETS_PATH, "X_val_normalized.npy"), X_val)
np.save(os.path.join(DATASETS_PATH, "y_val_normalized.npy"), y_val)
print("[OK] val (normalized): saved X/y as .npy")

del X_val, y_val
gc.collect()


15,334/15,334
[OK] val (normalized): saved X/y as .npy


44

In [23]:
# =============================================================================
# STEP 5c — Standardize TEST with the TRAIN mean/std (same logic as VAL).
# =============================================================================

norm_params = np.load(os.path.join(DATASETS_PATH, "pair_normalization_params.npz"), allow_pickle=True)
mean, std = norm_params["mean"], norm_params["std"]

X_test = np.load(os.path.join(DATASETS_PATH, "X_test.npy"))
y_test = np.load(os.path.join(DATASETS_PATH, "y_test.npy"))

X_test = transform_standardize(X_test, mean, std, chunk_size=100_000, std_eps=1e-8, verbose=True)

np.save(os.path.join(DATASETS_PATH, "X_test_normalized.npy"), X_test)
np.save(os.path.join(DATASETS_PATH, "y_test_normalized.npy"), y_test)
print("[OK] test (normalized): saved X/y as .npy")

del X_test, y_test
gc.collect()


15,438/15,438
[OK] test (normalized): saved X/y as .npy


44

## 8. STEP 6 — Final sanity checks on the normalized splits

Same structure as the H5 pipeline's final sanity check (shapes, NaN/Inf, standardization mean~0/std~1 via `compute_feature_stats`), adapted to this task's **4-class** (FF/FT/TF/TT) label space instead of the H5 pipeline's binary one, and extended with an `event_id` shape/consistency check specific to the pair dataset. `X`/`y` are reloaded with `load_mmap` (blockwise / memmap, no full-size load into RAM), consistent with the rest of the pipeline.

In [24]:
# =============================================================================
# STEP 6 — Sanity checks on each normalized split: shapes, 4-class balance,
# NaN/Inf, mean/std. All blockwise/memmap: no full-size array loaded into RAM.
# =============================================================================

def sanity_check_pair_split(name, directory, x_name, y_name, event_id_name,
                             expected_n=None, feature_dim=None, chunk_size=100_000):
    print(f"\n=== CHECK: {name} ===")

    X = load_mmap(directory, x_name)
    y = load_mmap(directory, y_name)
    event_id = np.load(os.path.join(directory, event_id_name))

    n_samples, n_features = X.shape

    # --- 1) Consistent shapes ---
    print(f"Shape X: {X.shape} | Shape y: {y.shape} | Shape event_id: {event_id.shape}")
    assert X.shape[0] == y.shape[0] == event_id.shape[0], \
        "MISMATCH: X, y and event_id have different n_samples!"
    if expected_n is not None:
        assert n_samples == expected_n, f"MISMATCH: expected {expected_n}, found {n_samples}"
    if feature_dim is not None:
        assert n_features == feature_dim, f"MISMATCH: expected {feature_dim} features, found {n_features}"

    # --- 2) 4-class (FF/FT/TF/TT) balance ---
    y_full = np.array(y)
    print("Class balance:")
    for idx in sorted(INDEX_TO_LABEL):
        count = int(np.sum(y_full == idx))
        frac = count / y_full.shape[0] if y_full.shape[0] else 0.0
        print(f"  {INDEX_TO_LABEL[idx]:>2}: {count:>10}  ({100.0 * frac:.3f}%)")
    assert set(np.unique(y_full)) <= set(LABEL_INDEX_MAP.values()), \
        "y contains label indices outside LABEL_INDEX_MAP!"

    # --- 3) NaN/Inf check, blockwise ---
    n_nan = 0
    n_inf = 0
    for start in range(0, n_samples, chunk_size):
        end = min(start + chunk_size, n_samples)
        block = X[start:end]
        n_nan += np.isnan(block).sum()
        n_inf += np.isinf(block).sum()
    print(f"Total NaN: {n_nan} | Total Inf: {n_inf}")
    assert n_nan == 0 and n_inf == 0, "NaN/Inf FOUND in X!"

    # --- 4) Standardization check: mean~0, std~1 per feature ---
    mean_check, std_check = compute_feature_stats(X, name=name, batch_size=chunk_size, verbose=True)

    return X, y, event_id  # memmap / plain array, doesn't occupy RAM until indexed


feature_dim = len(feature_names)

X_train_chk, y_train_chk, event_id_train_chk = sanity_check_pair_split(
    "TRAIN", DATASETS_PATH, "X_train_normalized.npy", "y_train_normalized.npy",
    "event_id_train.npy", feature_dim=feature_dim)
X_val_chk, y_val_chk, event_id_val_chk = sanity_check_pair_split(
    "VAL", DATASETS_PATH, "X_val_normalized.npy", "y_val_normalized.npy",
    "event_id_val.npy", feature_dim=feature_dim)
X_test_chk, y_test_chk, event_id_test_chk = sanity_check_pair_split(
    "TEST", DATASETS_PATH, "X_test_normalized.npy", "y_test_normalized.npy",
    "event_id_test.npy", feature_dim=feature_dim)

print("\n[OK] All checks passed.")



=== CHECK: TRAIN ===
Shape X: (71696, 21) | Shape y: (71696,) | Shape event_id: (71696,)
Class balance:
  FF:      53760  (74.983%)
  FT:         57  (0.080%)
  TF:      17444  (24.331%)
  TT:        435  (0.607%)
Total NaN: 0 | Total Inf: 0
TRAIN -> mean range [-0.000000, 0.000000] | std range [1.000000, 1.000000]

=== CHECK: VAL ===
Shape X: (15334, 21) | Shape y: (15334,) | Shape event_id: (15334,)
Class balance:
  FF:      11536  (75.232%)
  FT:          9  (0.059%)
  TF:       3698  (24.116%)
  TT:         91  (0.593%)
Total NaN: 0 | Total Inf: 0
VAL   -> mean range [-0.013685, 0.028612] | std range [0.950805, 1.072657]

=== CHECK: TEST ===
Shape X: (15438, 21) | Shape y: (15438,) | Shape event_id: (15438,)
Class balance:
  FF:      11572  (74.958%)
  FT:          6  (0.039%)
  TF:       3754  (24.317%)
  TT:        106  (0.687%)
Total NaN: 0 | Total Inf: 0
TEST  -> mean range [-0.014287, 0.025821] | std range [0.995859, 1.095826]

[OK] All checks passed.


## 9. *(Optional)* Wrap the splits in `JetDataset` / `DataLoader`

Ready-to-train `DataLoader`s for the FF/FT/TF/TT pair classifier (objective 3.1), built on top of the normalized splits checked in STEP 6, reusing `JetDataset` from `flavour_tag_ml.data` unmodified.

In [ ]:
# =============================================================================
# (OPTIONAL) Wrap each normalized split in a JetDataset / DataLoader.
# =============================================================================

BATCH_SIZE = 512

train_dataset = JetDataset(X_train_chk, y_train_chk)
val_dataset = JetDataset(X_val_chk, y_val_chk)
test_dataset = JetDataset(X_test_chk, y_test_chk)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"train_loader: {len(train_dataset)} samples, {len(train_loader)} batches")
print(f"val_loader:   {len(val_dataset)} samples, {len(val_loader)} batches")
print(f"test_loader:  {len(test_dataset)} samples, {len(test_loader)} batches")


train_loader: 377567 samples, 738 batches
val_loader:   80923 samples, 159 batches
test_loader:  81083 samples, 159 batches


/content/flavour_tagging/src/flavour_tag_ml/data.py:47: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  self.X = torch.as_tensor(X, dtype=torch.float32)
